In [8]:
import re
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print('Libraries loaded successfully!')

Libraries loaded successfully!


In [9]:
DATA_FILE = 'candidates.csv'
candidates = pd.read_csv(DATA_FILE)
    
print('Dataset loaded successfully!')
print('Number of candidates:', len(candidates))
display(candidates.head())

Dataset loaded successfully!
Number of candidates: 20


,CandidateID,Name,Skills,Education,ExperienceYears,TargetRole,Email
0,C001,Ananya Sharma,"Python, SQL, Machine Learning, NLP, Pandas, Sc...",B.Tech Computer Science,3,ML Engineer,ananya@example.com
1,C002,Rahul Kumar,"Java, Spring Boot, SQL, REST API, Git, Docker",B.Tech IT,4,Backend Developer,rahul@example.com
2,C003,Priya Nair,"Python, NLP, Spacy, NLTK, Machine Learning, SQL",M.Tech AI,2,NLP Engineer,priya@example.com
3,C004,Arjun Menon,"HTML, CSS, JavaScript, React, Node.js, MongoDB",B.Tech CSE,3,Full Stack Developer,arjun@example.com
4,C005,Sneha Patel,"Python, Excel, SQL, Power BI, Tableau, Statistics",B.Sc Data Science,2,Data Analyst,sneha@example.com


In [10]:
SKILL_LIST = [
    'python', 'java', 'c++', 'c#', 'sql', 'machine learning', 'deep learning',
    'nlp', 'nltk', 'spacy', 'scikit-learn', 'tensorflow', 'pytorch',
    'pandas', 'numpy', 'matplotlib', 'power bi', 'tableau', 'excel',
    'html', 'css', 'javascript', 'typescript', 'react', 'node.js',
    'mongodb', 'postgresql', 'aws', 'azure', 'docker', 'kubernetes',
    'terraform', 'linux', 'networking', 'siem', 'splunk', 'wireshark',
    'git', 'rest api', 'xgboost', 'opencv', 'transformers', 'bert'
]

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9+#.\s-]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

def extract_skills(text):
    cleaned = clean_text(text)
    return sorted({skill for skill in SKILL_LIST if skill in cleaned})

print('NLP functions ready!')

NLP functions ready!


In [11]:
job_description = '''
Looking for an NLP Engineer with Python, NLP, NLTK or spaCy,
Machine Learning, SQL and experience with text classification.
'''

job_skills = extract_skills(job_description)

print('Extracted Job Skills:')
for skill in job_skills:
    print('-', skill)

Extracted Job Skills:
- machine learning
- nlp
- nltk
- python
- spacy
- sql


In [12]:
def rank_candidates(job_description, candidates):
    candidate_text = candidates['Skills'].fillna('').map(clean_text)
    job_text = clean_text(job_description)

    corpus = list(candidate_text) + [job_text]
    vectorizer = TfidfVectorizer(ngram_range=(1, 2), stop_words='english')
    matrix = vectorizer.fit_transform(corpus)
    similarities = cosine_similarity(matrix[-1], matrix[:-1]).flatten()

    results = candidates.copy()
    results['SimilarityScore'] = similarities * 100
    job_skill_set = set(extract_skills(job_description))
    
    results['MatchedSkills'] = results['Skills'].apply(
        lambda x: ', '.join(sorted(job_skill_set & set(extract_skills(x))))
    )
    results['MatchedSkillCount'] = results['MatchedSkills'].apply(
        lambda x: len([s for s in str(x).split(', ') if s])
    )

    return results.sort_values(
        ['SimilarityScore', 'MatchedSkillCount'],
        ascending=False
    ).reset_index(drop=True)

ranked = rank_candidates(job_description, candidates)

print('Candidate ranking completed!')

Candidate ranking completed!


In [13]:
display_columns = [
    'CandidateID', 'Name', 'TargetRole', 'ExperienceYears',
    'MatchedSkills', 'SimilarityScore', 'Email'
]

result = ranked[display_columns].copy()
result['SimilarityScore'] = result['SimilarityScore'].round(2)

print('Recommended Candidates')
display(result)

Recommended Candidates


,CandidateID,Name,TargetRole,ExperienceYears,MatchedSkills,SimilarityScore,Email
0,C003,Priya Nair,NLP Engineer,2,"machine learning, nlp, nltk, python, spacy, sql",42.19,priya@example.com
1,C019,Nithya Krishnan,NLP Engineer,2,"nlp, nltk, python, spacy, sql",41.74,nithya@example.com
2,C013,Aishwarya R,ML Engineer,2,"machine learning, nlp, python, sql",27.76,aishwarya@example.com
3,C001,Ananya Sharma,ML Engineer,3,"machine learning, nlp, python, sql",18.44,ananya@example.com
4,C017,Lakshmi Devi,Data Scientist,2,"machine learning, python, sql",13.09,lakshmi@example.com
5,C009,Divya Iyer,NLP Engineer,3,"nlp, python, sql",8.03,divya@example.com
6,C006,Vikram Singh,AI Engineer,4,python,3.69,vikram@example.com
7,C012,Naveen Kumar,Backend Developer,3,"python, sql",2.49,naveen@example.com
8,C011,Keerthi S,Data Analyst,1,"python, sql",2.46,keerthi@example.com
9,C015,Farah Khan,Business Analyst,2,"python, sql",2.39,farah@example.com


In [14]:
top_candidate = ranked.iloc[0]

print('TOP RECOMMENDATION')
print('Name:', top_candidate['Name'])
print('Role:', top_candidate['TargetRole'])
print('Similarity Score:', round(top_candidate['SimilarityScore'], 2), '%')
print('Matched Skills:', top_candidate['MatchedSkills'])

TOP RECOMMENDATION
Name: Priya Nair
Role: NLP Engineer
Similarity Score: 42.19 %
Matched Skills: machine learning, nlp, nltk, python, spacy, sql
